In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

from src.models.multimodal.pipeline_convnextcbam_clip_pvd import MultiModalPipelineConvNeXtCBAMCLIPPVD

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Config

In [3]:
VAL_ROOT = PROJECT_ROOT / "data" / "processed" / "PlantDocSplited_depth_AUG" / "validation"

IMAGE_CKPT_PATH = PROJECT_ROOT / "archive" / "best_model.pth"
FUSION_CKPT_PATH = PROJECT_ROOT / "archive" / "multimodal" / "multimodal_pvd_classifier_state_dict.pth"

NUM_CLASSES = 28
BATCH_SIZE = 16
IMG_SIZE = 224
NUM_WORKERS = 2

SAVE_DIR = PROJECT_ROOT / "archive" / "multimodal" / "validation_results"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = SAVE_DIR / "classification_report.csv"
CM_PATH = SAVE_DIR / "confusion_matrix.csv"
PRED_PATH = SAVE_DIR / "predictions.csv"


Helper

In [4]:
def class_name_to_text(class_name: str) -> str:
    # Có thể đổi template này nếu muốn
    clean_name = class_name.replace("_", " ").strip()
    return f"a photo of {clean_name}"

Dataset

In [5]:
class ValidationMultimodalDataset(Dataset):
    def __init__(self, root: Path, transform=None):
        self.base_dataset = datasets.ImageFolder(root=str(root), transform=transform)
        self.classes = self.base_dataset.classes
        self.class_to_idx = self.base_dataset.class_to_idx
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        image_path, _ = self.base_dataset.samples[idx]

        class_name = self.idx_to_class[label]
        text = class_name_to_text(class_name)

        return {
            "image": image,
            "text": text,
            "label": label,
            "image_path": image_path,
            "class_name": class_name,
        }


def collate_fn(batch):
    images = torch.stack([item["image"] for item in batch], dim=0)
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    image_paths = [item["image_path"] for item in batch]
    class_names = [item["class_name"] for item in batch]

    return {
        "image": images,
        "text": texts,
        "label": labels,
        "image_path": image_paths,
        "class_name": class_names,
    }

Main

In [6]:
def main():
    print("Using device:", device)
    print("VAL_ROOT:", VAL_ROOT)
    print("IMAGE_CKPT_PATH:", IMAGE_CKPT_PATH)
    print("FUSION_CKPT_PATH:", FUSION_CKPT_PATH)

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    dataset = ValidationMultimodalDataset(root=VAL_ROOT, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    print("Number of validation samples:", len(dataset))
    print("Classes:", dataset.classes)

    model = MultiModalPipelineConvNeXtCBAMCLIPPVD(
        image_ckpt_path=IMAGE_CKPT_PATH,
        fusion_ckpt_path=FUSION_CKPT_PATH,
        num_classes=NUM_CLASSES,
        clip_model_name="ViT-L-14",
        clip_pretrained="openai",
        image_input_dim=1024,
        text_input_dim=768,
        proj_dim=512,
        proj_hidden_dim=768,
        pvd_hidden_dim=768,
        cls_hidden_dim=1024,
        dropout=0.3,
        normalize_projection=False,
        device=device,
    ).to(device)

    model.eval()

    y_true = []
    y_pred = []
    prediction_rows = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            texts = batch["text"]
            labels = batch["label"].to(device, non_blocking=True)

            logits = model(images, texts)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

            for i in range(len(texts)):
                true_idx = labels[i].item()
                pred_idx = preds[i].item()

                prediction_rows.append({
                    "image_path": batch["image_path"][i],
                    "text": texts[i],
                    "true_label_idx": true_idx,
                    "pred_label_idx": pred_idx,
                    "true_class_name": dataset.idx_to_class[true_idx],
                    "pred_class_name": dataset.idx_to_class[pred_idx],
                    "correct": int(true_idx == pred_idx),
                })

    acc = accuracy_score(y_true, y_pred)
    print(f"Validation Accuracy: {acc:.4f}")

    report = classification_report(
        y_true,
        y_pred,
        target_names=dataset.classes,
        digits=4,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred)

    report_df = pd.DataFrame(report).transpose()
    cm_df = pd.DataFrame(cm, index=dataset.classes, columns=dataset.classes)
    pred_df = pd.DataFrame(prediction_rows)

    report_df.to_csv(REPORT_PATH, encoding="utf-8-sig")
    cm_df.to_csv(CM_PATH, encoding="utf-8-sig")
    pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")

    print("Saved classification report to:", REPORT_PATH)
    print("Saved confusion matrix to:", CM_PATH)
    print("Saved predictions to:", PRED_PATH)


if __name__ == "__main__":
    main()

Using device: cuda
VAL_ROOT: /media/data3/users/luongdth/MulCo-PlantNet/data/processed/PlantDocSplited_depth_AUG/validation
IMAGE_CKPT_PATH: /media/data3/users/luongdth/MulCo-PlantNet/archive/best_model.pth
FUSION_CKPT_PATH: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal/multimodal_pvd_classifier_state_dict.pth
Number of validation samples: 333
Classes: ['Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf', 'Bell_pepper_leaf', 'Bell_pepper_leaf_spot', 'Blueberry_leaf', 'Cherry_leaf', 'Corn_Gray_leaf_spot', 'Corn_leaf_blight', 'Corn_rust_leaf', 'Peach_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight', 'Raspberry_leaf', 'Soyabean_leaf', 'Squash_Powdery_mildew_leaf', 'Strawberry_leaf', 'Tomato_Early_blight_leaf', 'Tomato_Septoria_leaf_spot', 'Tomato_leaf', 'Tomato_leaf_bacterial_spot', 'Tomato_leaf_late_blight', 'Tomato_leaf_mosaic_virus', 'Tomato_leaf_yellow_virus', 'Tomato_mold_leaf', 'Tomato_two_spotted_spider_mites_leaf', 'grape_leaf', 'grape_leaf_black_rot']


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/media/data3/users/luongdth/anaconda3/envs/gr1/lib/python3.12/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Validation Accuracy: 0.8649
Saved classification report to: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal/validation_results/classification_report.csv
Saved confusion matrix to: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal/validation_results/confusion_matrix.csv
Saved predictions to: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal/validation_results/predictions.csv
